# Model Training: XGBoost 3-Month-Ahead Climate Risk Forecasting

## 1. Project Objective & Context
This notebook trains and validates an **XGBoost** model to forecast tropical Pacific ocean climate risk tiers **3 months in advance** to support supply chain resiliency against El Niño / La Niña disruptions.

### Target Variables
- **`target_risk` (Classification):** 5 NOAA ONI risk tiers (*Extreme Cool*, *Moderate Cool*, *Neutral*, *Moderate Warm*, *Extreme Warm*).
- **`target_ssta` (Regression):** Continuous Sea Surface Temperature Anomaly (°C) 3 months in the future.

### Benchmarks to Beat (Validation 1994–1996)
1. **Majority-Class (Dummy) Baseline:** **0.132 Macro F1** (always predicts "Neutral"; 0.00 recall on extremes).
2. **Persistence Baseline:** **0.353 Macro F1** (predicts that the class in 3 months equals today's class).

> **Core Objective:** The model must decisively outperform both the naive dummy baseline (0.132) and the domain-specific persistence baseline (0.353).

---

## 2. Documented Experiment Suite (Progression of Approaches)
This notebook documents our systematic engineering journey across 5 distinct model iterations:
1. **Experiment 1 (Direct XGBClassifier - Full 36 Features):** Multiclass log-loss with balanced class weights.
2. **Experiment 2 (Direct XGBRegressor - Full 36 Features):** Predicts continuous temperature, then applies static NOAA tiers.
3. **Experiment 3 (Coordinate & Noise Ablation - 21 Features):** Removes `lat` and `lon` to eliminate geographic overfitting from buoy network expansion.
4. **Experiment 4 (Residual Tendency Forecaster):** Predicts the 3-month temperature change $\Delta = \text{target\_ssta} - \text{ssta}$ anchored to today's temperature.
5. **Experiment 5 (Winning Combined Solution):** Physical residual forecaster + threshold calibration to counter regression-to-the-mean shrinkage.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
from xgboost import XGBClassifier, XGBRegressor

print(f"XGBoost version: {xgb.__version__}")

# Styling
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 3. Load Data & Split Verification
- **Training Set (1980–1993):** 1,988 site-months.
- **Validation Set (1994–1996):** 1,529 site-months.
- **Test Set (1997–1998):** Blind holdout (not used for model selection).

In [ ]:
DATA_DIR = "../data/processed"
TRAIN_PATH = os.path.join(DATA_DIR, "features-train.csv")
VAL_PATH = os.path.join(DATA_DIR, "features-val.csv")

train_raw = pd.read_csv(TRAIN_PATH)
val_raw = pd.read_csv(VAL_PATH)

# Filter to rows where target labels are present
train = train_raw.dropna(subset=["target_risk", "target_ssta", "current_risk"]).copy()
val = val_raw.dropna(subset=["target_risk", "target_ssta", "current_risk"]).copy()

print(f"Train samples: {len(train):,} ({train['ym'].min()} to {train['ym'].max()})")
print(f"Val samples:   {len(val):,} ({val['ym'].min()} to {val['ym'].max()})")

# Class distribution comparison
dist_df = pd.DataFrame({
    "Train Count": train["target_risk"].value_counts(),
    "Train %": (train["target_risk"].value_counts(normalize=True) * 100).round(2),
    "Val Count": val["target_risk"].value_counts(),
    "Val %": (val["target_risk"].value_counts(normalize=True) * 100).round(2),
}).loc[["Extreme Cool", "Moderate Cool", "Neutral", "Moderate Warm", "Extreme Warm"]]

dist_df

## 4. Benchmark Baselines (The Lower Bounds)
1. **Majority-Class (Dummy):** Always predicts "Neutral" (58% of training data).
2. **Persistence Baseline:** Assumes ocean risk tier in 3 months will equal today's tier (`target_risk = current_risk`).

In [ ]:
RISK_CLASSES = ["Extreme Cool", "Moderate Cool", "Neutral", "Moderate Warm", "Extreme Warm"]
CLASS_TO_INT = {c: i for i, c in enumerate(RISK_CLASSES)}
INT_TO_CLASS = {i: c for i, c in enumerate(RISK_CLASSES)}

# Storage for full results tracking
results = []

# 1. Dummy Baseline
dummy_preds = ["Neutral"] * len(val)
dummy_f1 = f1_score(val["target_risk"], dummy_preds, average="macro", zero_division=0)
dummy_acc = accuracy_score(val["target_risk"], dummy_preds)

# 2. Persistence Baseline
persist_preds = val["current_risk"]
persist_f1 = f1_score(val["target_risk"], persist_preds, average="macro", zero_division=0)
persist_acc = accuracy_score(val["target_risk"], persist_preds)

results.append({
    "Model / Strategy": "Majority-Class (Dummy)",
    "Features": 0,
    "Macro F1": round(dummy_f1, 4),
    "Accuracy": round(dummy_acc, 4),
    "Extreme Cool F1": 0.000,
    "Extreme Warm F1": 0.000,
    "Beats Persistence?": False,
})

results.append({
    "Model / Strategy": "Persistence Baseline",
    "Features": 1,
    "Macro F1": round(persist_f1, 4),
    "Accuracy": round(persist_acc, 4),
    "Extreme Cool F1": round(f1_score(val["target_risk"] == "Extreme Cool", persist_preds == "Extreme Cool"), 3),
    "Extreme Warm F1": round(f1_score(val["target_risk"] == "Extreme Warm", persist_preds == "Extreme Warm"), 3),
    "Beats Persistence?": False,
})

print(f"Dummy Baseline Macro F1:       {dummy_f1:.4f} | Accuracy: {dummy_acc:.4f}")
print(f"Persistence Baseline Macro F1: {persist_f1:.4f} | Accuracy: {persist_acc:.4f}")

## 5. Feature Sets Definition
We systematically define and compare three feature sets:
- **`FEATURES_FULL` (36 features):** The full set with local wind lags, humidity, and calendar harmonics.
- **`FEATURES_TRIMMED` (23 features):** Pruned based on `docs/EDA_SUMMARY.md` (removes noise features).
- **`FEATURES_PHYSICAL` (16 features):** Pruned of spatial coordinates (`lat`, `lon`), forcing the model to learn universal thermodynamics rather than memorizing early mooring sites.

In [ ]:
FEATURES_TRIMMED = [
    "ssta", "ssta_lag1", "ssta_lag2", "ssta_lag3", "ssta_lag6",
    "ssta_roll3", "ssta_roll6", "ssta_change1", "ssta_change3",
    "basin_ssta", "nino34_ssta", "nino34_ssta_roll3", "basin_zwind_anom", "west_pac_zwind_anom",
    "air_temp_anom", "air_temp_anom_lag1", "air_temp_anom_lag2", "air_temp_anom_lag3", "air_temp_anom_lag6",
    "lat", "lon",
    "zonal_wind_anom", "mer_wind_anom",
]

FEATURES_NOISE = [
    "humidity", "month_sin", "month_cos",
    "zonal_wind_anom_lag1", "zonal_wind_anom_lag2", "zonal_wind_anom_lag3", "zonal_wind_anom_lag6",
    "zonal_wind_anom_roll3", "zonal_wind_anom_roll6",
    "mer_wind_anom_lag1", "mer_wind_anom_lag2", "mer_wind_anom_lag3", "mer_wind_anom_lag6",
]

FEATURES_FULL = FEATURES_TRIMMED + FEATURES_NOISE

# Pure thermodynamic features without coordinates
FEATURES_PHYSICAL = [
    "ssta", "ssta_lag1", "ssta_lag2", "ssta_lag3", "ssta_roll3", "ssta_change1", "ssta_change3",
    "basin_ssta", "nino34_ssta", "nino34_ssta_roll3", "basin_zwind_anom", "west_pac_zwind_anom",
    "air_temp_anom", "air_temp_anom_lag1", "zonal_wind_anom", "mer_wind_anom"
]

print(f"Full Feature Set:     {len(FEATURES_FULL)} features")
print(f"Trimmed Feature Set:  {len(FEATURES_TRIMMED)} features")
print(f"Physical Feature Set: {len(FEATURES_PHYSICAL)} features")

## 6. Model Experiments & Diagnostics

### Standard NOAA Classification Thresholds
$$
\text{Class} = \begin{cases} 
\text{Extreme Warm} & \text{if } a \ge 1.5 \\
\text{Moderate Warm} & \text{if } 0.5 \le a < 1.5 \\
\text{Neutral} & \text{if } -0.5 < a < 0.5 \\
\text{Moderate Cool} & \text{if } -1.5 < a \le -0.5 \\
\text{Extreme Cool} & \text{if } a \le -1.5
\end{cases}
$$

In [ ]:
def classify_noaa_static(a):
    if pd.isna(a): return np.nan
    if a >= 1.5:   return "Extreme Warm"
    if a >= 0.5:   return "Moderate Warm"
    if a <= -1.5:  return "Extreme Cool"
    if a <= -0.5:  return "Moderate Cool"
    return "Neutral"

def record_result(name, num_feats, y_pred):
    f1 = f1_score(val["target_risk"], y_pred, average="macro", zero_division=0)
    acc = accuracy_score(val["target_risk"], y_pred)
    ec_f1 = f1_score(val["target_risk"] == "Extreme Cool", y_pred == "Extreme Cool", zero_division=0)
    ew_f1 = f1_score(val["target_risk"] == "Extreme Warm", y_pred == "Extreme Warm", zero_division=0)
    entry = {
        "Model / Strategy": name,
        "Features": num_feats,
        "Macro F1": round(f1, 4),
        "Accuracy": round(acc, 4),
        "Extreme Cool F1": round(ec_f1, 3),
        "Extreme Warm F1": round(ew_f1, 3),
        "Beats Persistence?": f1 > persist_f1,
    }
    results.append(entry)
    print(f"{name:<38} | Macro F1: {f1:.4f} | Acc: {acc:.4f} | Beats Persistence: {f1 > persist_f1}")
    return f1

# -----------------------------------------------------
# Experiment 1: XGBClassifier (Full 36 Features)
# -----------------------------------------------------
sample_weights = compute_sample_weight("balanced", train["target_risk"])
clf_full = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, eval_metric="mlogloss")
clf_full.fit(train[FEATURES_FULL], train["target_risk"].map(CLASS_TO_INT), sample_weight=sample_weights)
preds_clf_full = pd.Series(clf_full.predict(val[FEATURES_FULL])).map(INT_TO_CLASS)
record_result("Exp 1: XGBClassifier (Full 36)", len(FEATURES_FULL), preds_clf_full)

# -----------------------------------------------------
# Experiment 2: XGBRegressor (Full 36 Features -> Static Tiers)
# -----------------------------------------------------
reg_full = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
reg_full.fit(train[FEATURES_FULL], train["target_ssta"])
preds_reg_full = pd.Series(reg_full.predict(val[FEATURES_FULL])).map(classify_noaa_static)
record_result("Exp 2: XGBRegressor (Full 36 -> Tiers)", len(FEATURES_FULL), preds_reg_full)

# -----------------------------------------------------
# Experiment 3: XGBClassifier (No Coordinates / lat-lon)
# -----------------------------------------------------
clf_phys = XGBClassifier(n_estimators=50, max_depth=2, learning_rate=0.03, reg_lambda=10.0, random_state=42, eval_metric="mlogloss")
clf_phys.fit(train[FEATURES_PHYSICAL], train["target_risk"].map(CLASS_TO_INT), sample_weight=sample_weights)
preds_clf_phys = pd.Series(clf_phys.predict(val[FEATURES_PHYSICAL])).map(INT_TO_CLASS)
record_result("Exp 3: XGBClassifier (Physics 16, No Geo)", len(FEATURES_PHYSICAL), preds_clf_phys)

### Experiment 4 & 5: The Winning Combined Solution (Residual Forecaster)

#### Why Residual Forecasting Solves the Failure Modes:
1. **Anchored to Reality:** Rather than predicting the future ocean from a blank slate, the model predicts the **temperature change (tendency $\Delta$)**:
   $$\Delta = \text{target\_ssta} - \text{ssta}$$
   $$\widehat{\text{future\_ssta}} = \text{ssta}_{\text{today}} + \widehat{\Delta}$$
2. **Eliminating Spatial Overfitting:** Uses `FEATURES_PHYSICAL` (excluding `lat` and `lon`), forcing the trees to rely on thermal momentum and winds.
3. **Threshold Calibration:** Because multi-month regressions naturally shrink toward zero (regression to the mean), we apply calibrated risk thresholds that restore sensitivity to rare extremes.

In [ ]:
# 1. Define target as the 3-month delta
delta_train = train["target_ssta"] - train["ssta"]

# 2. Train regularized residual booster
best_xgb_reg = XGBRegressor(
    n_estimators=40,
    max_depth=2,
    learning_rate=0.04,
    reg_lambda=10.0,
    random_state=42,
)
best_xgb_reg.fit(train[FEATURES_PHYSICAL], delta_train)

# 3. Predict delta and construct future SST anomaly
pred_delta_val = best_xgb_reg.predict(val[FEATURES_PHYSICAL])
pred_ssta_val = val["ssta"] + pred_delta_val

# 4. Calibrated NOAA mapping
def classify_calibrated(a):
    if a >= 1.0:   return "Extreme Warm"
    if a >= 0.35:  return "Moderate Warm"
    if a <= -1.1:  return "Extreme Cool"
    if a <= -0.4:  return "Moderate Cool"
    return "Neutral"

winning_preds = pred_ssta_val.map(classify_calibrated)

# Record both uncalibrated and calibrated residual models
record_result("Exp 4: Residual Model (Uncalibrated)", len(FEATURES_PHYSICAL), pred_ssta_val.map(classify_noaa_static))
best_f1 = record_result("Exp 5: Winning Residual (Calibrated)", len(FEATURES_PHYSICAL), winning_preds)

print("\n=== WINNING MODEL DETAILED CLASSIFICATION REPORT ===")
print(classification_report(val["target_risk"], winning_preds, zero_division=0))

## 7. Model Comparison Summary Table
Direct comparison of all approaches against the Dummy and Persistence baselines.

In [ ]:
summary_df = pd.DataFrame(results)
summary_df

## 8. How the Winning Model Works (With a Real Example)

### Step-by-Step Inference Mechanism
1. **Input:** Today's buoy reading ($	ext{ssta} = -1.25^\circ\text{C}$), recent rolling means, and basin wind indicators.
2. **Residual Prediction:** The tree evaluates physical momentum features and predicts the 3-month tendency $\widehat{\Delta} = -0.18^\circ\text{C}$ (indicating continued cooling).
3. **Forecast Recomposition:**
   $$\widehat{\text{target\_ssta}} = \text{ssta} + \widehat{\Delta} = -1.25 + (-0.18) = -1.43^\circ\text{C}$$
4. **Threshold Mapping:** Since $-1.43 \le -1.1$, the final forecasted tier is **Extreme Cool**.

In [ ]:
# Real validation example demonstrating the winning model
sample_idx = val[val["target_risk"] == "Extreme Cool"].index[10]
sample_row = val.loc[sample_idx]

# Slice directly as a DataFrame with double brackets and enforce float dtype
sample_features = val.loc[[sample_idx], FEATURES_PHYSICAL].astype(float)
sample_current_ssta = float(sample_row["ssta"])
sample_pred_delta = float(best_xgb_reg.predict(sample_features)[0])
sample_pred_future = sample_current_ssta + sample_pred_delta
sample_predicted_class = classify_calibrated(sample_pred_future)

print(f"Buoy Site:               {sample_row['site_id']} (Month: {sample_row['ym']})")
print(f"Current Anomaly (ssta):  {sample_current_ssta:+.3f} °C (Class: {sample_row['current_risk']})")
print(f"Predicted 3-Month Delta: {sample_pred_delta:+.3f} °C")
print(f"Forecasted Future SSTA:  {sample_pred_future:+.3f} °C")
print(f"Forecasted Risk Tier:    {sample_predicted_class}")
print(f"Actual Future Risk Tier: {sample_row['target_risk']} (Actual Future SSTA: {float(sample_row['target_ssta']):+.3f} °C)")

## 9. Multi-Dimensional Diagnostics & Spatial Fairness
Evaluating the winning model across two critical dimensions:
1. **Confusion Matrix:** Verifying non-zero sensitivity to rare extremes.
2. **Spatial Breakdown:** Verifying performance across Pacific longitude bands:
   - **Western Pacific** ($140^\circ\text{E} - 180^\circ$)
   - **Central Pacific** ($180^\circ - 140^\circ\text{W}$)
   - **Eastern Pacific** ($140^\circ\text{W} - 95^\circ\text{W}$)

In [ ]:
# 1. Confusion Matrix
cm = confusion_matrix(val["target_risk"], winning_preds, labels=RISK_CLASSES)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=RISK_CLASSES, yticklabels=RISK_CLASSES)
plt.title(f"Confusion Matrix: Winning Residual Model (Macro F1 = {best_f1:.4f})")
plt.xlabel("Predicted Risk Tier")
plt.ylabel("Actual Risk Tier")
plt.tight_layout()
plt.show()

# 2. Spatial Fairness
val_diag = val.copy()
val_diag["pred_risk"] = winning_preds.values

def assign_region(lon):
    if lon > 0 or lon == 180:
        return "West Pacific (140E-180)"
    elif lon < -140:
        return "Central Pacific (180-140W)"
    else:
        return "East Pacific (140W-95W)"

val_diag["region"] = val_diag["lon"].apply(assign_region)

print("=== SPATIAL FAIRNESS BREAKDOWN ===")
for region, r_df in val_diag.groupby("region"):
    r_f1 = f1_score(r_df["target_risk"], r_df["pred_risk"], average="macro", zero_division=0)
    r_acc = accuracy_score(r_df["target_risk"], r_df["pred_risk"])
    print(f"{region:<28} | Samples: {len(r_df):<4} | Macro F1: {r_f1:.3f} | Accuracy: {r_acc:.3f}")

## 10. Model Visualizations & Physical Interpretability
We focus on three high-impact visualizations that explain model behavior and operational utility:
1. **Feature Importance (Physical Drivers):** Which thermodynamic variables drive the 3-month tendency ($\Delta$)?
2. **Forecasted vs. Actual SSTA with NOAA Risk Tiers:** Scatter analysis demonstrating how calibrated boundaries capture extreme tails without variance collapse.
3. **Time-Series Forecast Tracking on a Benchmark Mooring Site:** Chronological tracking of ground truth vs. persistence vs. XGBoost 3-month forecast across 1994–1996.

In [ ]:
# 1. Feature Importance for the 16 Physical Drivers
fi = pd.Series(best_xgb_reg.feature_importances_, index=FEATURES_PHYSICAL).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(fi.index, fi.values, color="steelblue", edgecolor="black", alpha=0.85)
plt.title("Physical Drivers of 3-Month Anomaly Tendency (XGBoost Gain)", fontsize=13, fontweight="bold")
plt.xlabel("Normalized Feature Importance (Gain)", fontsize=11)
plt.ylabel("Thermodynamic Feature", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Forecasted vs. Actual SSTA with Shaded Risk Class Agreement Zones
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 8))

# Shaded 2D Agreement Zones where (Actual Class == Forecasted Class):
# Rectangle signature: ((x_min, y_min), width, height)
ax.add_patch(patches.Rectangle((-3.0, -3.0), 1.5, 1.9, facecolor="#a6c8e0", alpha=0.35, label="Extreme Cool Agreement"))
ax.add_patch(patches.Rectangle((-1.5, -1.1), 1.0, 0.7, facecolor="#c2e5e7", alpha=0.45, label="Moderate Cool Agreement"))
ax.add_patch(patches.Rectangle((-0.5, -0.4), 1.0, 0.75, facecolor="#dcdcdc", alpha=0.35, label="Neutral Agreement"))
ax.add_patch(patches.Rectangle((0.5, 0.35), 1.0, 0.65, facecolor="#fde2b4", alpha=0.45, label="Moderate Warm Agreement"))
ax.add_patch(patches.Rectangle((1.5, 1.0), 1.5, 2.0, facecolor="#f8b4b4", alpha=0.35, label="Extreme Warm Agreement"))

# Validation scatter points
ax.scatter(val["target_ssta"], pred_ssta_val, alpha=0.40, color="navy", edgecolor="none", s=24, zorder=3, label="Validation Buoy-Months")
ax.plot([-3, 3], [-3, 3], "k--", linewidth=1.3, zorder=4, label="Perfect Forecast Line (1:1)")

# Threshold lines on Y-axis (Forecasted Boundaries: -1.1, -0.4, 0.35, 1.0)
for y_th, c in zip([-1.10, -0.40, 0.35, 1.00], ["darkblue", "cornflowerblue", "orange", "firebrick"]):
    ax.axhline(y_th, color=c, linestyle=":", linewidth=1.3, zorder=2)

# Threshold lines on X-axis (Actual NOAA Boundaries: -1.5, -0.5, 0.5, 1.5)
for x_th, c in zip([-1.50, -0.50, 0.50, 1.50], ["darkblue", "cornflowerblue", "orange", "firebrick"]):
    ax.axvline(x_th, color=c, linestyle=":", linewidth=1.3, zorder=2)

ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.8, 2.8)
ax.set_xlabel("Actual Future SSTA 3 Months Ahead (°C) [NOAA Ground Truth Tiers]", fontsize=11)
ax.set_ylabel("Forecasted Future SSTA (°C) [Calibrated Model Tiers]", fontsize=11)
ax.set_title("Forecasted vs. Actual 3-Month SSTA with Shaded Class Agreement Zones", fontsize=12, fontweight="bold")
ax.legend(loc="upper left", fontsize=8.5, frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Time-Series Tracking at Benchmark Mooring Site (-2°S, 110°W)
benchmark_site = "-2_-110"
site_df = val[val["site_id"] == benchmark_site].sort_values("ym").copy()
site_pred_delta = best_xgb_reg.predict(site_df[FEATURES_PHYSICAL])
site_df["pred_future_ssta"] = site_df["ssta"] + site_pred_delta

plt.figure(figsize=(12, 5))
plt.plot(site_df["ym"], site_df["target_ssta"], "k-o", linewidth=2.2, label="Actual SSTA (3 Months Ahead)")
plt.plot(site_df["ym"], site_df["pred_future_ssta"], "b--s", linewidth=2.0, label="XGBoost Residual Forecast")
plt.plot(site_df["ym"], site_df["ssta"], "gray", linestyle=":", linewidth=1.5, label="Persistence Baseline (Today SSTA)")

plt.axhline(-1.5, color="darkblue", linestyle="--", alpha=0.5, label="NOAA Extreme Cool Threshold (-1.5°C)")
plt.title(f"Chronological Forecast Tracking at Site {benchmark_site} (1994–1996 Validation)", fontsize=13, fontweight="bold")
plt.xlabel("Validation Month (Year-Month)", fontsize=11)
plt.ylabel("Sea Surface Temp Anomaly (°C)", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.legend(loc="lower left", frameon=True, fontsize=9.5)
plt.tight_layout()
plt.show()

## 11. Test-Driven Development (TDD) Validation Assertions
Automated assertion that the winning model strictly beats both baselines.

In [ ]:
# Assertion 1: Beats naive dummy baseline
assert best_f1 > dummy_f1, f"TDD Assertion FAILED: {best_f1:.4f} <= {dummy_f1:.4f}"

# Assertion 2: Decisively beats domain-specific persistence baseline
assert best_f1 > persist_f1, f"TDD Assertion FAILED: {best_f1:.4f} <= {persist_f1:.4f}"

print(f"✅ ALL TDD TESTS PASSED!")
print(f"   Winning Model Macro F1 = {best_f1:.4f} > Persistence Baseline ({persist_f1:.4f})")
print(f"   Winning Model Macro F1 = {best_f1:.4f} > Dummy Baseline       ({dummy_f1:.4f})")

## 12. Model Export for Downstream Integration
Exports the trained XGBoost booster to universal JSON format for team cross-comparison (vs LightGBM) and SHAP explainability.

In [ ]:
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)
export_path = os.path.join(MODELS_DIR, "best_xgboost_residual_model.json")
best_xgb_reg.save_model(export_path)
print(f"Model successfully exported to: {export_path}")